In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
import ipywidgets as widgets
from IPython.display import display

### Load Data

In [2]:
df_fact = pd.read_csv("C:/Users/mazen/OneDrive/Desktop/steamUserFact.csv").drop_duplicates()
df_user = pd.read_csv("C:/Users/mazen/OneDrive/Desktop/userDim.csv")
df_game = pd.read_csv("C:/Users/mazen/OneDrive/Desktop/gameDim.csv", encoding="latin-1")

### Clean and Filter

In [3]:
df_game['price_clean'] = pd.to_numeric(df_game['price'], errors='coerce').fillna(0)
df_fact = df_fact[df_fact["playtime_hours"] > 2]
df_merged = df_fact.merge(df_game, on='game_id', how='inner')

### Build User Features


In [4]:
user_stats = df_merged.groupby('steam_id').agg(
    total_games=('game_id', 'count'),
    total_playtime=('playtime_hours', 'sum'),
    avg_price=('price_clean', 'mean')
).reset_index()

X = user_stats.drop('steam_id', axis=1).fillna(0)
scaler = StandardScaler()
user_feat_x = scaler.fit_transform(X)

### K-Means Clustering

In [5]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
user_stats['cluster'] = kmeans.fit_predict(user_feat_x)

cluster_profiles = {
    0: 'Mainstream Action Gamer',
    1: 'Free-to-Play Casual',
    2: 'Broad / Power User',
    3: 'Independent Enthusiast'
}
user_stats['segment_label'] = user_stats['cluster'].map(cluster_profiles)

### Association Rules

In [6]:
df_transactions = df_merged[['steam_id', 'game_name']].merge(
    user_stats[['steam_id', 'segment_label']], on='steam_id')

### Dictionary

In [7]:
segment_rules = {}

print("Generating association rules per segment.")

for segment in df_transactions['segment_label'].unique():
    segment_data = df_transactions[df_transactions['segment_label'] == segment]
    
    transactions = segment_data.groupby('steam_id')['game_name'].apply(list).tolist()
    
    te = TransactionEncoder()
    encoded = te.fit_transform(transactions)
    bask = pd.DataFrame(encoded, columns=te.columns_)
    
    fp_itemset = fpgrowth(bask, min_support=0.015, use_colnames=True)
    
    if not fp_itemset.empty:
        rules = association_rules(fp_itemset, metric='lift', min_threshold=1.2)
        rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
        segment_rules[segment] = rules
    else:
        segment_rules[segment] = pd.DataFrame()

print("Rule generation complete.")

Generating association rules per segment.
Rule generation complete.


### Interactive Widget

In [ ]:
all_games = df_merged['game_name'].dropna().unique().tolist()
all_segments = list(cluster_profiles.values())

segment_dropdown = widgets.Dropdown(
    options=all_segments,
    description='Gamer Profile:',
    style={'description_width': 'initial'},
    layout={'width': '350px'}
)

text_input = widgets.Combobox(
    description='Target Game:',
    placeholder='Type a game name...',
    options=all_games,
    ensure_option=False,
    style={'description_width': 'initial'},
    layout={'width': '350px'}
)

button = widgets.Button(
    description='Get Segmented Recommendations', 
    button_style='info', 
    layout={'width': '250px', 'margin': '0 0 0 20px'}
)
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        user_search = text_input.value.strip()
        selected_segment = segment_dropdown.value
        
        if not user_search:
            print("Please enter a game name.")
            return
            
        print(f"\nTargeting segment: {selected_segment}")
        print(f"Base Game: '{user_search}'")
        print("-" * 50)
        
        rules = segment_rules.get(selected_segment, pd.DataFrame())
        
        if rules.empty:
            print("No association rules found for this specific user segment.")
            return
            
        search_set = frozenset([user_search])
        
        matching_rules = rules[rules['antecedents'] == search_set]
        
        if len(matching_rules) > 0:
            top_rules = matching_rules.sort_values(by=['lift', 'confidence'], ascending=False).head(8)
            
            print(f"Other {selected_segment}s who played '{user_search}' also loved:\n")
            
            for index, row in top_rules.iterrows():
                recommended_items = ", ".join(list(row['consequents']))
                confidence = round(row['confidence'] * 100, 1)
                lift = round(row['lift'], 2)
                
                print(f"   {recommended_items}")
                print(f"   (Confidence: {confidence}% | Lift: {lift}x)")
        else:
            print(f"\nNo strong segment-specific patterns found for '{user_search}'.")
            print("Try a more popular game within this demographic, or check spelling.")

button.on_click(on_button_clicked)

ui = widgets.VBox([
    widgets.HBox([segment_dropdown, text_input]), 
    widgets.Box([button], layout=widgets.Layout(margin='10px 0 0 0'))
])
display(ui, output)

Output()